# Illinois walkthrough — handle, State AGR, online vs retail

Official source: [IGB sports reports](https://igb.illinois.gov/sports-wagering/sports-reports.html).

Two different official CSVs:

1. **All Wagering Activity → Sport Detail** — handle by licensee and channel
2. **Completed Events → Tax Summary** — **State AGR** and **State Tax**

State AGR is adjusted revenue. We store it in `adjusted_revenue` and do **not** copy it into
`gross_revenue`. Retail rows are used to reconcile official `Total` lines, then excluded from
the primary `online_sports_betting` dataset.

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from variant_gaming.common import project_root
from variant_gaming.states.illinois import (
    join_handle_and_revenue,
    parse_sport_detail_handle,
    parse_tax_summary,
)
from variant_gaming.storage import connect, default_db_path

ROOT = project_root()
handle_text = (ROOT / "tests/fixtures/IL/sample_sport_detail.csv").read_text(encoding="utf-8")
tax_text = (ROOT / "tests/fixtures/IL/sample_tax_summary.csv").read_text(encoding="utf-8")
print(handle_text.splitlines()[0])
print(tax_text.splitlines()[0])

"All Wagering Activity Sport Detail"
"Completed Events Tax Summary"


## Clean handle, then tax, then join

In [2]:
handle_df, handle_check = parse_sport_detail_handle(handle_text)
print("handle max abs reconciling difference", handle_check["difference"].abs().max())
handle_df.head()

handle max abs reconciling difference 7.450580596923828e-09


,channel,operator,handle
48,retail,815 Entertainment,748828.57
49,retail,"Alton Casino, LLC",763812.91
50,retail,"Casino Queen, Inc.",1864539.00
51,retail,Elgin Riverboat Resort,818670.75
52,retail,"Fairmount Park, Inc.",104300.25


In [3]:
tax_df = parse_tax_summary(tax_text)
print("negative AGR rows", int((tax_df["adjusted_revenue"] < 0).sum()))
tax_df.head()

negative AGR rows 2


,channel,operator,adjusted_revenue,tax
0,retail,815 Entertainment,97109.20,19421.84
1,retail,"Alton Casino, LLC",0.00,0.00
2,retail,"Casino Queen, Inc.",321577.21,64315.44
3,retail,Elgin Riverboat Resort,107659.95,21531.99
4,retail,"Fairmount Park, Inc.",-8992.49,-1798.50


In [4]:
joined = join_handle_and_revenue(handle_df, tax_df)
online = joined[joined["channel"] == "online"]
print("online operators", len(online), "retail operators", int((joined["channel"]=="retail").sum()))
online.sort_values("operator").head()

online operators 16 retail operators 16


,channel,operator,handle,adjusted_revenue,tax
0,online,815 Entertainment,3.135424e+07,2481791.86,496358.37
1,online,"Alton Casino, LLC",1.006278e+08,6703577.45,1675894.36
2,online,"Casino Queen, Inc.",4.334164e+08,32773118.25,13109247.30
3,online,Elgin Riverboat Resort,4.299337e+07,3237869.35,809467.34
4,online,FHR-Illinois LLC,8.233489e+06,144834.16,28966.83


Retail exists in the official file. It is **not** labeled `online_sports_betting` in the
database. The collector upserts online operator rows only.

SQLite uses a unique key of state, vertical, channel, operator, row type, period, and source hash.
Re-runs update the same key; they do not wipe New York.

In [5]:
conn = connect(default_db_path(ROOT))
try:
    il = __import__("pandas").read_sql_query(
        "SELECT channel, row_type, COUNT(*) AS n FROM gaming_results WHERE state_code='IL' GROUP BY 1, 2",
        conn,
    )
except Exception:
    il = __import__("pandas").DataFrame()
conn.close()
il

,channel,row_type,n
0,online,official_statewide_total,76
1,online,operator,815
